# 03. RFM (Recency, Frequency, Monetary) Customer Analysis
**E-Commerce Customer Intelligence & Churn Prediction Platform**

### Methodology:
- **Recency ($R$):** Days elapsed since the customer's last purchase before cutoff snapshot.
- **Frequency ($F$):** Total count of completed orders.
- **Monetary ($M$):** Cumulative net spend across all transactions.
- We assign 1-5 quintile scores using statistical rank division and construct composite RFM tiers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load engineered customer features
df_feat = pd.read_csv("../data/processed/olist/customer_rfm.csv")
df_rfm = df_feat[['customer_id', 'recency_days', 'order_frequency', 'monetary_total']].copy()
df_rfm.columns = ['customer_id', 'recency', 'frequency', 'monetary']

# Quintile scoring (1 to 5)
df_rfm['r_score'] = pd.qcut(df_rfm['recency'].rank(method='first', ascending=False), 5, labels=[1, 2, 3, 4, 5]).astype(int)
df_rfm['f_score'] = pd.qcut(df_rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
df_rfm['m_score'] = pd.qcut(df_rfm['monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
df_rfm['rfm_score'] = df_rfm['r_score'] * 100 + df_rfm['f_score'] * 10 + df_rfm['m_score']

print("RFM Summary Statistics:")
print(df_rfm[['recency', 'frequency', 'monetary']].describe())

### 3.1 RFM Segment Assignment

In [ ]:
def assign_segment(r):
    if r['r_score'] >= 4 and r['f_score'] >= 4 and r['m_score'] >= 4:
        return 'Champions & VIPs'
    elif r['r_score'] >= 3 and r['f_score'] >= 3:
        return 'Loyal Customers'
    elif r['r_score'] >= 4 and r['f_score'] <= 2:
        return 'New / Promising'
    elif r['r_score'] <= 2 and r['m_score'] >= 4:
        return 'At-Risk High Spenders'
    elif r['r_score'] <= 2 and r['f_score'] <= 2:
        return 'Hibernating / Lost'
    else:
        return 'Needs Attention'

df_rfm['rfm_segment'] = df_rfm.apply(assign_segment, axis=1)

plt.figure(figsize=(10, 4))
df_rfm['rfm_segment'].value_counts().plot(kind='barh', color='#2ca02c')
plt.title("Distribution of RFM Customer Segments", fontsize=14, fontweight='bold')
plt.xlabel("Number of Customers")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Strategic Takeaways:
- Champions & VIPs represent high lifetime value and require high-touch loyalty engagement.
- At-Risk High Spenders require urgent win-back before permanent churn occurs.